# Day 2 · Gold standards & evaluation

*Day 2 — Linguistic Data Analysis II*

### How to use this notebook

This is your **single submission for the day**. It has three parts:

- **Part A · Tutorial** — load a gold standard → evaluate a **frozen** set of model predictions → read precision / recall / F1 + a confusion matrix → inspect the errors, on an easy-to-judge task (CEFR sentence level).
- **Part B · Corpus Lab** — build a gold standard yourself — annotate ~20 sentences by hand in a Google Sheet, measure your agreement, adjudicate, and import it back as canonical JSON.
- **Part C · Corpus Lab** — code the evaluation metrics (precision, recall, F1, Cohen's κ) by hand, then check them against scikit-learn.

You only edit the cells marked **✏️ YOU EDIT**. Cells marked **🔧 Library cell** are pre-written — run them, don't change them.

➡️ Work top to bottom. When you're done, **Runtime → Run all**, then **File → Download → Download `.ipynb`** and submit that file.

## Part A · Tutorial — the pipeline on CEFR-SP

The task (CEFR sentence level, A1–C2) is easy to judge on purpose — today the *mechanics* are the lesson, not the labeling.

::: {.callout-note}
## Today runs on *frozen* predictions — no API key, no live model
You met the live model on Day 1 and saw its answers change from run to run. When you're learning to *measure* quality, that wobble is just noise fighting the lesson — so today the model's predictions are **pre-computed and committed** to a file. You load them and everyone's precision / recall / F1 come out **identical every run**. You'll run the model yourself (with the key) from Day 3 on.
:::

### Setup — run this first

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Imports + the LLM backend. No pip install needed in Colab.
import json, re, urllib.request, os
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

MODEL_ID = "gemini-3.1-flash-lite"   # pinned model for the reproducible (API) backend

def _resolve_gemini_key():
    """Find a Gemini API key: Colab Secrets first (not auto-exported to env), then env."""
    try:
        from google.colab import userdata      # only exists in Colab
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass                                    # not in Colab, or secret not set
    return os.environ.get("GEMINI_API_KEY")

def _make_api_backend(key):
    """Reproducible backend: Gemini API with temperature=0 + a fixed seed."""
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    cfg = types.GenerateContentConfig(temperature=0, seed=42)
    return (lambda p: client.models.generate_content(
                model=MODEL_ID, contents=p, config=cfg).text,
            f"Gemini API ({MODEL_ID}, temperature=0, seed=42)")

# Prefer the API key when set (reproducible); else fall back to colab.ai (demo).
_key = _resolve_gemini_key()
if _key:
    generate_text, _backend = _make_api_backend(_key)
else:
    try:
        from google.colab import ai            # Colab's built-in Gemini — no key
        generate_text, _backend = (lambda p: ai.generate_text(p)), "Colab Gemini (demo, non-reproducible)"
    except ImportError:
        raise RuntimeError(
            "No LLM backend found. Run this notebook in Google Colab (free built-in "
            "Gemini, no key needed), or set GEMINI_API_KEY — in Colab via the Secrets "
            "panel, or as an environment variable when running locally. "
            "See resources/tools/gemini-api-key.md.")

# CEFR-SP gold set (72 sentences, 12 per level), fetched from the course repo.
GOLD_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/cefr_sentences.json"
LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
PREDICTIONS_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/predictions_day2.json"   # frozen model predictions
print(f"Setup done. LLM backend: {_backend}. scikit-learn ready.")

In [ ]:
#@title 🔧 Library cell: load_gold(url_or_path) → gold { display-mode: "form" }
def load_gold(url_or_path):
    """Read the canonical gold JSON: [{'id','text','label'}, ...]."""
    if str(url_or_path).startswith("http"):
        raw = urllib.request.urlopen(url_or_path).read().decode("utf-8")
        gold = json.loads(raw)
    else:
        gold = json.loads(open(url_or_path, encoding="utf-8").read())
    print(f"Loaded {len(gold)} items. First one:", gold[0])
    return gold

In [ ]:
#@title 🔧 Library cell: run_prompt(prompt, gold) → predictions { display-mode: "form" }
def _extract_level(text):
    """Pull the first A1/A2/B1/B2/C1/C2 out of the model's reply."""
    m = re.search(r"\b([ABC][12])\b", str(text).upper())
    return m.group(1) if m else "??"

def run_prompt(prompt, gold):
    """Send each item's `text` to the LLM via {text}, collect predicted labels."""
    predictions = []
    for i, item in enumerate(gold, 1):
        reply = generate_text(prompt.format(text=item["text"]))
        predictions.append(_extract_level(reply))
        if i % 12 == 0:
            print(f"  ...{i}/{len(gold)} done")
    print(f"Got {len(predictions)} predictions.")
    return predictions

In [ ]:
#@title 🔧 Library cell: evaluate(gold, predictions) → P/R/F1 + confusion matrix { display-mode: "form" }
def evaluate(gold, predictions):
    y_true = [item["label"] for item in gold]
    y_pred = predictions
    print(classification_report(y_true, y_pred, labels=LEVELS, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=LEVELS)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LEVELS, yticklabels=LEVELS)
    plt.xlabel("Predicted"); plt.ylabel("Gold"); plt.title("Confusion matrix")
    plt.tight_layout(); plt.show()

In [ ]:
#@title 🔧 Library cell: show_errors(gold, predictions) → misclassified table { display-mode: "form" }
def show_errors(gold, predictions):
    rows = [{"id": g["id"], "gold": g["label"], "pred": p, "text": g["text"]}
            for g, p in zip(gold, predictions) if g["label"] != p]
    print(f"{len(rows)} of {len(gold)} wrong.")
    return pd.DataFrame(rows)

In [ ]:
#@title 🔧 Library cell: save_predictions / load_predictions { display-mode: "form" }
def save_predictions(predictions, path):
    """Freeze a list of predicted labels to JSON."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f)
    print(f"Saved {len(predictions)} predictions to {path}")

def load_predictions(url_or_path):
    """Read a frozen predictions list — a committed URL or a local path."""
    if str(url_or_path).startswith("http"):
        raw = urllib.request.urlopen(url_or_path).read().decode("utf-8")
        predictions = json.loads(raw)
    else:
        predictions = json.loads(open(url_or_path, encoding="utf-8").read())
    print(f"Loaded {len(predictions)} frozen predictions.")
    return predictions

### Step 1 — Load the gold standard

Notice the shape: every dataset this week is `{"id", "text", "label"}`. That is the *only* data shape you have to learn.

In [ ]:
gold = load_gold(GOLD_URL)

### Step 2 — The prompt, and the frozen predictions it produced

Here is the prompt we sent the model — `{text}` is where each sentence gets slotted in. We ran it **once** over the gold set and committed the predictions, so today you load that frozen file rather than call the model. (From Day 3 you'll run prompts like this yourself.)

In [ ]:
# The prompt used to produce the frozen predictions (shown for reference — not run today):
PROMPT = """You are an expert rater of English sentence difficulty using the CEFR scale.
Classify the sentence into exactly one of: A1, A2, B1, B2, C1, C2.
Answer with the level only.

Sentence: {text}"""

# Load the pre-computed predictions (same order as `gold`):
predictions = load_predictions(PREDICTIONS_URL)

### Step 3 — Read the evaluation

Per-level precision / recall / F1 (plus a macro average), then the confusion matrix.

**Brace yourself: overall accuracy is about 39%.** Before you conclude the model is useless, look at *how* it is wrong. Roughly 97% of its answers are within one level of the gold label — only two sentences in the whole set miss by two, and nothing misses by more. A model guessing at random would scatter A1s against C2s. This one doesn't. It has the right idea and imprecise thresholds, which is a very different failure from not understanding the task.

Now read the confusion matrix **down the columns** — how often the model *says* each level, rather than how often it's right. The gold set is balanced, 12 sentences per level, so an unbiased rater would use each label about 12 times. Check whether this one does. Ask yourself why a rater — human or machine — under pressure to judge something as fuzzy as "difficulty" might drift toward the middle of a scale and avoid committing to the extremes. You will meet this tendency again in your own annotation work.

Because the predictions are frozen, these numbers are the same every time you run — that's the point.

In [ ]:
evaluate(gold, predictions)

### Step 4 — Error analysis

There are 44 misses here — too many to read one by one, and you don't need to. Skim a dozen, then look specifically at the rows where the gold label is **C2** or **A1**: the ends of the scale are where this model disagrees with the gold most often, so that is where the interesting arguments are.

For each miss you do read, ask: is the **gold** defensible, or is this a genuinely borderline sentence? *"Is the disagreement the model's fault or the scheme's?"* is the heart of annotation work — and it is the question your mini-project has to answer honestly.

In [ ]:
errors = show_errors(gold, predictions)
errors.head(15)     # ...or errors[errors["gold"] == "C2"] to see the hard end

## Part B · Corpus Lab — build a gold standard yourself

So far the gold labels were handed to you. Now you make some. You will annotate ~20 sentences **by hand in a Google Sheet**, with your partner labelling the same sentences independently, then bring the result back into Python as a canonical gold file — the same `{"id", "text", "label"}` shape you loaded in Part A.

::: {.callout-note}
## Why a spreadsheet?
Annotation is a *judgement* task, not a coding task — a sheet is the fastest honest way to do it, and it is what most annotation projects actually use. The point is not the tool; it is that you feel how often two careful people disagree, and what it takes to resolve that into a single defensible label.
:::

In [ ]:
#@title 🔧 Library cell: Google Sheets annotation round-trip { display-mode: "form" }
ANNOTATION_HEADER = ["id", "text", "label_a", "label_b", "final", "notes"]

def _sheets_client():
    """Authorise gspread with your Google account (a pop-up asks for permission)."""
    from google.colab import auth
    import google.auth, gspread
    auth.authenticate_user()
    creds, _ = google.auth.default()
    return gspread.authorize(creds)

def create_annotation_sheet(title, items, labels):
    """Create a Sheet in YOUR Drive: one row per item, blank columns to label.
    `items` are {"id","text",...} dicts — any existing label is deliberately NOT
    copied across, so you annotate blind. Returns the sheet URL."""
    sheet = _sheets_client().create(title)
    worksheet = sheet.sheet1
    rows = [[item["id"], item["text"], "", "", "", ""] for item in items]
    worksheet.update([ANNOTATION_HEADER] + rows)
    worksheet.freeze(rows=1)
    print(f"Created '{title}' with {len(rows)} rows to annotate.")
    print("Allowed labels:", ", ".join(labels))
    print("Open it:", sheet.url)
    return sheet.url

def load_annotation_sheet(title):
    """Read your annotation sheet back as a list of row dicts."""
    rows = _sheets_client().open(title).sheet1.get_all_records()
    print(f"Read {len(rows)} rows from '{title}'.")
    return rows

def to_canonical(rows, labels, column="final"):
    """Turn annotation rows into canonical gold: [{"id","text","label"}, ...].
    Blank rows are skipped; labels outside `labels` are reported, not silently kept."""
    gold, blank, invalid = [], 0, []
    for row in rows:
        label = str(row.get(column, "")).strip()
        if not label:
            blank += 1
        elif label not in labels:
            invalid.append((row.get("id"), label))
        else:
            gold.append({"id": int(row["id"]), "text": str(row["text"]), "label": label})
    print(f"{len(gold)} usable · {blank} still blank · {len(invalid)} invalid")
    if invalid:
        print("  fix these in the sheet, then re-run:", invalid[:10])
    return gold

def annotator_agreement(rows, a="label_a", b="label_b"):
    """Percent agreement + Cohen's κ between the two annotator columns."""
    from sklearn.metrics import cohen_kappa_score
    pairs = [(str(r.get(a, "")).strip(), str(r.get(b, "")).strip()) for r in rows]
    pairs = [(x, y) for x, y in pairs if x and y]
    if not pairs:
        print("No rows where BOTH annotators have labelled. Nothing to compare yet.")
        return None
    percent = sum(x == y for x, y in pairs) / len(pairs)
    kappa = cohen_kappa_score([x for x, _ in pairs], [y for _, y in pairs])
    print(f"{len(pairs)} doubly-annotated · agreement {percent:.1%} · Cohen's κ {kappa:.3f}")
    return {"n": len(pairs), "percent_agreement": percent, "kappa": kappa}

def disagreements(rows, a="label_a", b="label_b"):
    """The rows your two annotators labelled differently — your adjudication list."""
    out = [r for r in rows
           if str(r.get(a, "")).strip() and str(r.get(b, "")).strip()
           and str(r[a]).strip() != str(r[b]).strip()]
    print(f"{len(out)} rows to adjudicate. Agree on a `final` label for each in the sheet.")
    return pd.DataFrame(out)

def compare_to_published(gold, published):
    """How often does YOUR final label match the published gold, item by item?"""
    lookup = {item["id"]: item["label"] for item in published}
    shared = [(g["label"], lookup[g["id"]]) for g in gold if g["id"] in lookup]
    if not shared:
        print("No overlapping ids — did you keep the ids the sheet gave you?")
        return None
    agree = sum(mine == theirs for mine, theirs in shared)
    print(f"{agree}/{len(shared)} match the published label ({agree / len(shared):.1%})")
    return pd.DataFrame([{"id": g["id"], "yours": g["label"], "published": lookup[g["id"]],
                          "text": g["text"]}
                         for g in gold if g["id"] in lookup
                         and g["label"] != lookup[g["id"]]])

### B1 — Draw ~20 sentences to annotate   ✏️ YOU EDIT

We sample from the same CEFR set you just evaluated — but the sheet gets **only the ids and the sentences**, never the labels. That way your annotation is genuinely independent, and at the end you can check it against the published gold.

In [ ]:
import random

N_ITEMS = 20            # ✏️ how many sentences to annotate
random.seed(42)         # fixed seed = the same sample every run

to_annotate = random.sample(gold, N_ITEMS)
print(f"{len(to_annotate)} sentences drawn. First:", to_annotate[0]["text"])

### B2 — Create your annotation sheet

This creates a Sheet **in your own Drive** and prints a link. Colab will ask permission to access your Google account — that is the `authenticate_user()` pop-up; accept it with your Tohoku account.

In the sheet, one of you fills **`label_a`** and the other fills **`label_b`** — *without looking at each other's column*. Leave `final` blank for now. Use `notes` for anything you found hard to decide; those notes are useful evidence later.

In [ ]:
SHEET_NAME = "lda2_day2_annotation"     # ✏️ rename if you like

create_annotation_sheet(SHEET_NAME, to_annotate, LEVELS)

::: {.callout-important}
## Stop here and go annotate
Open the link above and label all ~20 sentences in **both** annotator columns before running the next cell. Come back when the sheet is filled in.
:::

### B3 — Read it back and measure your agreement

How often did the two of you pick the same level? **Cohen's κ** corrects that for the agreement you would expect by chance alone — you will code it yourself in Part C.

In [ ]:
rows = load_annotation_sheet(SHEET_NAME)
annotator_agreement(rows)

### B4 — Adjudicate the disagreements

Every row below is one your two annotators saw differently. Talk each one through, decide a single label, and type it into the **`final`** column in the sheet. For the rows you already agreed on, `final` is just that agreed label.

Keep track of *why* you disagreed — is the sentence genuinely borderline, or is the scheme underspecified? That distinction is the heart of your mini-project report.

In [ ]:
disagreements(rows)

### B5 — Import it as a gold standard   ✏️ YOU EDIT

Once every row has a `final` label, read the sheet again and convert it to the canonical form. `to_canonical` refuses anything that is not one of your allowed labels, so typos surface here rather than silently corrupting your gold set.

In [ ]:
rows = load_annotation_sheet(SHEET_NAME)       # re-read, now with `final` filled in
my_gold = to_canonical(rows, LEVELS, column="final")
my_gold[:3]

### B6 — How does your gold compare with the published gold?

The CEFR-SP labels came from language-education professionals, and we kept only sentences where two of them agreed. Where you differ from them, you are not simply *wrong* — but each difference is worth a look.

In [ ]:
compare_to_published(my_gold, gold)

### B7 — Save your gold set to your Drive

Your gold set belongs in **your** Drive, not the course repo. See [Housing your data in Google Drive](../resources/tools/google-drive-data.md).

In [ ]:
# ✏️ Uncomment in Colab to save:
# from google.colab import drive; drive.mount("/content/drive")
# with open("/content/drive/MyDrive/my_gold_day2.json", "w", encoding="utf-8") as f:
#     json.dump(my_gold, f, ensure_ascii=False, indent=2)
# print("saved", len(my_gold), "items")

## Part C · Corpus Lab — code the metrics from scratch

`evaluate()` above printed precision, recall, F1 and a confusion matrix *for* you. Now you write those formulas yourself, from plain lists of labels — no imports, no numpy. Once you have coded them, the numbers scikit-learn reports in your final project will never be a black box.

Fill in each function (replace its `raise NotImplementedError(...)`), then run the **self-check** cell — it compares your functions against scikit-learn on a small example and prints ✅ / ❌ for each.

In [ ]:
# ✏️ YOU EDIT — implement each function. All take plain Python lists of labels.

def confusion_counts(gold, pred, label):
    """Return {"tp","fp","fn","tn"} for ONE label, treating it as the positive class.
        tp: gold IS label AND pred IS label
        fp: gold is NOT label BUT pred IS label
        fn: gold IS label BUT pred is NOT label
        tn: gold is NOT label AND pred is NOT label
    Example (label="B2", gold=["A1","B2","B2"], pred=["A1","B2","C1"]):
        {"tp": 1, "fp": 0, "fn": 1, "tn": 1}
    """
    # HINT: start tp=fp=fn=tn=0; loop `for g, p in zip(gold, pred):`;
    #       if/elif on whether g == label / p == label.
    raise NotImplementedError("Count tp, fp, fn, tn and return them in a dict.")


def precision(gold, pred, label):
    """tp / (tp + fp). Of all items CALLED `label`, what fraction really were?
    Return 0.0 if tp + fp == 0 (never divide by zero)."""
    raise NotImplementedError("Return tp / (tp + fp), or 0.0 if tp + fp == 0.")


def recall(gold, pred, label):
    """tp / (tp + fn). Of all items that TRULY were `label`, what fraction found?
    Return 0.0 if tp + fn == 0."""
    raise NotImplementedError("Return tp / (tp + fn), or 0.0 if tp + fn == 0.")


def f1(gold, pred, label):
    """Harmonic mean of precision and recall:
        2 * p * r / (p + r).   Return 0.0 if p + r == 0."""
    # HINT: reuse precision(...) and recall(...) above.
    raise NotImplementedError("Return the harmonic mean of precision and recall.")


def macro_f1(gold, pred, labels):
    """Plain average of f1(gold, pred, label) over every label in `labels`
    (every class counts equally, no matter how many items it has)."""
    raise NotImplementedError("Average f1 over every label in `labels`.")


def percent_agreement(a, b):
    """Fraction of positions where two annotators agree: a[i] == b[i]. 0.0–1.0."""
    raise NotImplementedError("Return the fraction of positions where a[i] == b[i].")


def cohen_kappa(a, b):
    """Agreement corrected for chance:  (p_o - p_e) / (1 - p_e).
        p_o = observed agreement = percent_agreement(a, b)
        p_e = chance agreement = sum over labels of prop_a(label) * prop_b(label),
              where prop_x(label) = (# times x used label) / N.
    Return 0.0 if 1 - p_e == 0."""
    # HINT: labels = set(a) | set(b); N = len(a); a.count(label) counts uses.
    raise NotImplementedError("Return (p_o - p_e) / (1 - p_e), or 0.0 if 1 - p_e == 0.")

In [ ]:
#@title 🔎 Self-check against scikit-learn — run me { display-mode: "form" }
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             cohen_kappa_score)

# A small fixture with imbalance and a few mistakes — enough to catch bugs.
LAB = ["A1", "A2", "B1", "B2"]
g = ["A1", "A1", "A2", "A2", "B1", "B1", "B1", "B2", "B2", "B2"]
p = ["A1", "A2", "A2", "A2", "B1", "B1", "B2", "B2", "B2", "A2"]
ann_a = ["A1", "A2", "B1", "B1", "B2", "A1", "A2", "B2", "B1", "A1"]
ann_b = ["A1", "A2", "B1", "B2", "B2", "A2", "A2", "B2", "B1", "A1"]

TOL, results = 1e-9, []
def _chk(name, got, exp):
    ok = abs(got - exp) < TOL
    results.append(ok)
    print(("✅" if ok else "❌"), f"{name:<22} yours={got:.6f}  sklearn={exp:.6f}")

for label in LAB:
    _chk(f"precision({label})", precision(g, p, label),
         precision_score(g, p, labels=[label], average="micro", zero_division=0))
    _chk(f"recall({label})", recall(g, p, label),
         recall_score(g, p, labels=[label], average="micro", zero_division=0))
    _chk(f"f1({label})", f1(g, p, label),
         f1_score(g, p, labels=[label], average="micro", zero_division=0))
_chk("macro_f1", macro_f1(g, p, LAB),
     f1_score(g, p, labels=LAB, average="macro", zero_division=0))
_chk("percent_agreement", percent_agreement(ann_a, ann_b),
     sum(1 for x, y in zip(ann_a, ann_b) if x == y) / len(ann_a))
_chk("cohen_kappa", cohen_kappa(ann_a, ann_b), cohen_kappa_score(ann_a, ann_b))

print("-" * 55)
print(f"All {len(results)} checks passed ✅  — your metrics match scikit-learn."
      if all(results) else
      f"{results.count(False)} of {len(results)} checks FAILED — fix and re-run.")

---
## ✅ Before you submit

1. **Runtime → Run all** and check every cell ran without error.
2. Tutorial outputs are visible (tables / charts / the model's answers).
3. Every Corpus Lab self-check prints ✅ (or your TODO answers are filled in).
4. **File → Download → Download `.ipynb`** and upload that one file.